[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/11_sft_grpo_unsloth.ipynb)

# 11 · GRPO on Qwen3 Base（Unsloth + W&B）——从共享 SFT 检查点 S1 开始

**先跑 notebook 15**（共享 SFT 检查点，只训练一次）。本 notebook 默认 `RUN_SFT=False`、`INIT_MODEL="auto"`：自动加载 15 推到 HF 的
`<你的用户名>/qwen3-4b-s1-sft`，只做 GRPO。结构参考 Unsloth 官方 [Qwen3 (4B) GRPO](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb)。

**X1b 里 GRPO 没用的原因，和这一版怎么改**（`experiments/results.md`）：
1. **起点没有推理可以强化**：X1b 的 SFT 回答是"背一段知识 + 答案"。→ S1 用带分步解答的科学数据训练，在 GPQA 题面下会逐个分析选项（notebook 15）。
2. **题库和 GPQA 不够像**：MMLU-Pro 偏本科、Dr.SCI 是数值题。→ `GRPO_DATA="science"`：一半是 **MegaScience 里留出的 4 选 1 研究生水平科学题**
   （和 GPQA 同一种题面；SFT 没见过它们的解答；只留两份解答答案一致的题），再加 20% MMLU-Pro STEM 和 30% Dr.SCI 数值题。
3. **训练量太小、每步太吵**：X1b 是 100 步 × 4 题。→ 120 步 × 16 题 × 8 个回答 = 1920 道题，学习率 1e-5（Dr.SCI / OLMo 3 全参是 1e-6，LoRA 放大 10 倍）。
4. **量不出差别**：单样本评测的噪声比 GRPO 的效果大。→ 留出验证集（同分布、不参与训练）训练前后各测一次；GPQA 每题采样 8 次，报告配对差值 ± 标准误。

论文里同等规模的参照（Dr.SCI，同一个 Qwen3-4B-Base）：SFT 后再做 RL，GPQA-D 提高 2–3 分；去掉太容易的题再 +2。所以**预期是 1–3 分**，要靠配对比较才看得出来。

**流程（照 OLMo 3：SFT → DPO → RL）**：15（SFT）→ 可选 12（DPO，推到 HF）→ 本 notebook（`INIT_MODEL` = S1 或 DPO 模型）。

**不用守着**：Run all 之后可以离开。adapter、每一段的结果、评测结果都存在 Google Drive（开头会弹一次授权）；断线后重新 Run all 会跳过已完成的评测和 GRPO 阶段；
全部跑完自动推到 HF 私有仓库并断开 runtime。改了配置想重跑某一格，直接重跑即可，不用重启（只有换模型 / 上下文长度才需要 Restart session，会有明确提示）。

| 模式 | 用途 | GRPO | 评测 |
|---|---|---|---|
| `quick` | 验证流程（T4 也行，0.6B；先用 15 的 quick 模式做出 0.6B 的 S1，或设 `RUN_SFT=True`） | 30 步 × 2 题 × 4 | Diamond 前 60 题，1 次采样 |
| `medium` | 1.7B 看变化（要 `RUN_SFT=True` 或自己的 `INIT_MODEL`） | 60 步 × 8 题 × 8 | Diamond，2 次采样 |
| `a100` | **4B 正式实验**（A100 40GB） | 120 步 × 16 题 × 8，两段；预筛 + 训练 + 验证约 2.5 小时（估） | Diamond + Main，≤8192，8 次采样（每次约 17 分钟） |

**生成预算**：排行榜的锁定评测是 8192，本 notebook 默认也用 8192。`GEN_BUDGET=16384` 只在参考分支用，
适合从思维链检查点开始的实验；这时结果里会多一列 `acc_flex_at_8192`（同样的回答按排行榜预算截断后的分数）。

⚠️ 快速评测**不是**排行榜分数（选项顺序与 lm-eval 不同）。正式分数：合并推到 HF 后用 `main` 分支的 `01_eval_gpqa.ipynb`。

In [1]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

name, memory.total [MiB], compute_cap
NVIDIA A100-SXM4-40GB, 40960 MiB, 8.0


In [2]:
# 克隆提交分支（数据准备 / 去污染 / 共用代码 scripts/recipe_utils.py 都在里面）
import os, sys
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
if os.path.basename(os.getcwd()) != "GPQA-eval":   # 重跑这一格时不要再进一层
    %cd GPQA-eval
!git pull -q
if os.path.abspath("scripts") not in sys.path:
    sys.path.insert(0, os.path.abspath("scripts"))
import recipe_utils
print("reloaded:", recipe_utils.reload_scripts())   # 改了 scripts/ 之后重跑这一格即可，不用重启 runtime

/content/GPQA-eval
reloaded: ['recipe_utils']


In [3]:
# 安装（约 3–5 分钟）。输出不隐藏：如果安装报错请看这里。
# 2026-11 实测可用：unsloth 2026.11.3、vllm 0.31、trl 1.15、transformers 5.17（A100）
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth vllm wandb {_numpy} {_pil}

Using Python 3.13.16 environment at: /usr
Resolved 228 packages in 1.07s
Prepared 92 packages in 17.52s
Uninstalled 13 packages in 978ms
Installed 92 packages in 281ms
 + agent-detector==2.0.0
 + anthropic==2.8.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + bitsandbytes==0.50.2
 + blake3==1.0.11
 + cbor2==6.3.0
 + compressed-tensors==0.17.0
 - cuda-core==0.7.0
 + cuda-core==1.3.0
 + cuda-tile==1.8.1
 + cut-cross-entropy==25.1.1
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dnspython==2.9.2
 + email-validator==2.4.0
 - fastapi==0.141.1
 + fastapi==0.136.3
 + fastapi-cli==0.0.34
 + fastapi-cloud-cli==0.28.0
 + fastar==0.12.0
 + fastsafetensors==0.5.0
 + flashinfer-python==0.7.0.post1
 + hf-transfer==0.1.9
 + humming-kernels==0.1.16
 + ijson==3.6.0
 + instanttensor==0.2.3
 + interegular==0.3.3
 + jmespath==1.1.0
 + json5==0.17.3
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.7.6
 - llvmlite==0.46.0
 + llvmlite==0.47.0
 + lm-format-enforcer==0.11.3
 + loguru==0.8.0
 + mcp==2.3.0
 + mcp-ty

In [4]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"   # Unsloth：GRPO 时让 vLLM 和训练共享显存
if torch.cuda.get_device_capability()[0] < 8:   # T4 不支持 FlashInfer 采样器
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
VERSIONS = {}
for p in ["unsloth", "vllm", "trl", "transformers", "torch", "peft", "wandb"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(GPU_NAME, GPU_MEM_GB, "GB"); print(VERSIONS)

NVIDIA A100-SXM4-40GB 39.5 GB
{'unsloth': '2026.11.4', 'vllm': '0.31.0', 'trl': '1.15.0', 'transformers': '5.17.0', 'torch': '2.13.0', 'peft': '0.21.2', 'wandb': '0.28.1'}


In [5]:
# Colab 左侧 🔑 Secrets：HF_TOKEN（必需；先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款）
#                      WANDB_API_KEY（强烈建议；没有就不记录 W&B）
for _k in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass
from huggingface_hub import login, whoami
login(token = os.environ.get("HF_TOKEN"))
HF_USER = whoami(token = os.environ.get("HF_TOKEN"))["name"]
print("HF user:", HF_USER, "|", "W&B key found" if os.environ.get("WANDB_API_KEY") else "No WANDB_API_KEY: W&B logging disabled")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF user: darcyyao | W&B key found


## 0. 配置

In [6]:
EXP_NAME     = "R2"                           # 实验编号：W&B run 名、报告里用、Drive 目录名。R1 = 在 60 步试跑检查点上训的那次（已崩溃）；换起点就换编号，不要沿用旧目录
MODE         = "a100"                         # "quick" | "medium" | "a100"
MODEL_NAME   = "unsloth/Qwen3-4B-Base"        # quick: "unsloth/Qwen3-0.6B-Base"；medium: "unsloth/Qwen3-1.7B-Base"
BASE_MODEL   = "Qwen/Qwen3-4B-Base"           # 提交时填的原始 base
LOAD_IN_4BIT = False
CHAT_STYLE   = "olmo"   # "olmo"（回答以 <|endoftext|> 结束，推荐）| "plain"（无特殊 token）| "chatml"（X1 的旧模板，只作对比）

# 流程开关
RUN_SFT         = False  # False = 从 INIT_MODEL 开始只做 GRPO（推荐）；True = 旧流程：在本 notebook 里先做一遍 X1b 的 SFT
INIT_MODEL      = "auto" # "auto" = notebook 15 的共享 SFT 检查点 f"{HF_USER}/qwen3-4b-s1-sft"；也可以填 notebook 12 的 DPO 模型或本地路径
ALLOW_UNVERIFIED_INIT = False  # False = 起点检查点必须带 handoff.json 且是训完的 a100 运行，否则停下（避免在试跑检查点上花几小时 GRPO）；确认无误后可设 True
PUSH_SFT_REPO   = None   # 只在 RUN_SFT=True 时用：SFT 后合并推到 HF 私有仓库
RUN_GRPO        = True
PUSH_FINAL_REPO = "auto" # "auto" = f"{HF_USER}/qwen3-4b-<EXP_NAME>-grpo"（私有；正式评测前在 HF 上改成公开）；None = 只存到持久目录
FINAL_ADAPTER   = None   # None = 合并最后一个阶段；"grpo_lora_s0"（或 RUN_SFT=True 时的 "sft_lora"）= 改为合并持久目录里的这个 adapter
USE_DRIVE       = True   # adapter、阶段结果、评测结果都存到 Google Drive：runtime 重置后不会丢
RESUME          = True   # 已经完成的阶段（评测、每一段 GRPO）直接读结果，不重跑；False = 全部重来
AUTO_DISCONNECT = True   # 全部跑完后自动断开 runtime（不再消耗 CU）。想留着调试就设 False
GEN_BUDGET      = None   # 评测的生成预算。None = 预设（a100: 8192，和排行榜一致）；16384 = 只在参考分支放宽，从思维链检查点（15 的 VARIANT="think"）开始时用
RUN_INIT_EVAL   = "after"  # 起点模型的 GPQA 评测（a100：Diamond + Main、每题 8 次，约 1 小时以上）什么时候做。"after" = GRPO 训完之后（训练结果先落盘，计算单元用完也不会白等）；True = 训练之前；False = 不做，最后就没有"起点 → GRPO"的配对比较。想省时间改这个开关，不要中途点停止
REUSE_INIT_EVAL = True   # True = 起点模型的 GPQA 分数直接用 notebook 15 测过的结果（Drive 上 15 的运行目录里，带逐题结果），这次不再评测起点；找不到或对不上就按 RUN_INIT_EVAL 评测
EVAL_SAMPLES    = None   # None = 预设（a100 每题采样 8 次；复用了 15 的起点评测时跟它一致，15 是每题 4 次）；填数字 = 固定用这个次数。4 次比 8 次省一半时间，标准误变大（Diamond 上约 ±2.4 分）
EVAL_CONFIGS    = None   # None = 预设（a100: Diamond + Main）；["gpqa_diamond"] = 只测 Diamond（Main 的题量是它的 2 倍多）
RUN_BASE_EVAL   = False  # base 已测过两次（X1 / X1b：Diamond 26、Main 24，回答不收尾所以要约 30 分钟）；INIT_MODEL 不为空时总是评测起点

# SFT 数据（--include/--exclude 同时匹配来源名和 domain；X1 里 Logic Puzzles 占 29%，现在去掉）
DOLCI_DATASET    = "allenai/Dolci-Instruct-SFT-7B"
DOLCI_INCLUDE    = ["science", "math", "gsm"]
DOLCI_EXCLUDE    = ["safety", "wildguard", "python", "code", "aya", "precise", "flan", "logic"]
THINK_DATASET    = "allenai/Dolci-Think-SFT-7B"
THINK_N          = 0            # >0：再从 Dolci-Think 抽这么多条思维链（长度 ≤ sft_max_tokens）
THINK_INCLUDE    = ["science", "math"]
EXTRA_SFT        = []           # 见 experiments/DATASETS.md，例如 [dict(dataset="MegaScience/TextbookReasoning", n=4000)]
SFT_SELECT       = "coverage"   # "random" | "coverage"（Dr.SCI 的 EESFT：按 4-gram 覆盖度从候选池里挑）
SFT_POOL_FACTOR  = 3            # coverage 时先抽 3 倍候选
COVERAGE_NORM    = "sqrt"       # "none" = 完全照论文（偏向长样本）；"sqrt" = 除以 √长度

# GRPO
GRPO_DATA         = "science"   # "science"（50% MegaScience 留出的 4 选 1 科学题 + 20% MMLU-Pro STEM + 30% Dr.SCI 数值题）| "mix"（X1b：MMLU-Pro + Dr.SCI）| "mmlu_pro_stem" | "drsci" | "sciq"
GRPO_FILTER       = True        # 每段训练前每题采样 4 次，只留答对 1–3 次的（去掉太易 / 太难的题）
CURRICULUM_ROUNDS = 2           # 2 = 两段：前一半较易（Dr.SCI 通过率 0.5–0.875），后一半较难（0.25–0.375），第二段用更新后的模型重新预筛
DRSCI_SUBJECTS    = ["physics", "chemistry", "biology"]
GRPO_LR           = 1e-5        # Dr.SCI / OLMo 3 全参 1e-6，LoRA 放大 10 倍；X1b 的 5e-6 × 100 步没有效果。验证集通过率不动可以试 2e-5
GRPO_MAX_COMPLETION = "auto"    # "auto" = 2 × 模型自己收尾长度的 99 分位（不超过预设上限）；也可以直接填整数
EPSILON_HIGH      = 0.28        # DAPO / OLMo 3 的 clip-higher
FORMAT_WEIGHT     = 0.5         # 格式分（正确性 +2 为主）
OVERLONG_CACHE    = 1024        # 软超长惩罚：最后 1024 token 里从 0 线性扣到 −1

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "gpqa-hackathon")
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY")

PRESETS = dict(
    quick = dict(max_seq_len=2048, gpu_util=0.7, lora_rank=32,
                 sft_dolci_n=200, sft_mcq_n=200, sft_max_tokens=1024, sft_steps=40, sft_lr=2e-4, sft_batch=4, sft_accum=2,
                 grpo_steps=30, grpo_prompts_per_step=2, num_gen=4, grpo_max_completion=512, filter_k=4, filter_max_tokens=512,
                 val_n=40, eval_configs=["gpqa_diamond"], eval_n=60, eval_max_tokens=1024, eval_samples=1, shuffle_buffer=1000),
    medium = dict(max_seq_len=8192, gpu_util=0.75, lora_rank=32,
                  sft_dolci_n=1000, sft_mcq_n=500, sft_max_tokens=4096, sft_steps=150, sft_lr=2e-4, sft_batch=4, sft_accum=2,
                  grpo_steps=60, grpo_prompts_per_step=8, num_gen=8, grpo_max_completion=2048, filter_k=4, filter_max_tokens=2048,
                  val_n=100, eval_configs=["gpqa_diamond"], eval_n=None, eval_max_tokens=4096, eval_samples=2, shuffle_buffer=5000),
    a100 = dict(max_seq_len=12288, gpu_util=0.8, lora_rank=64,
                sft_dolci_n=20000, sft_mcq_n=2000, sft_max_tokens=6144, sft_steps=-1, sft_lr=1e-4, sft_batch=4, sft_accum=4,
                grpo_steps=120, grpo_prompts_per_step=16, num_gen=8, grpo_max_completion=6144, filter_k=4, filter_max_tokens=3072,
                val_n=400, eval_configs=["gpqa_diamond", "gpqa_main"], eval_n=None, eval_max_tokens=8192, eval_samples=8,
                shuffle_buffer=20000),
)
Q = dict(PRESETS[MODE])
if EVAL_SAMPLES: Q["eval_samples"] = EVAL_SAMPLES
if EVAL_CONFIGS: Q["eval_configs"] = list(EVAL_CONFIGS)
if GEN_BUDGET:   # 放宽预算：上下文 = 预算 + 最长题面；回答长的模型，GRPO 的回答上限和预筛长度也跟着放宽
    Q.update(eval_max_tokens = GEN_BUDGET, max_seq_len = max(Q["max_seq_len"], GEN_BUDGET + 4096))
    if GEN_BUDGET > 8192:
        Q.update(grpo_max_completion = max(Q["grpo_max_completion"], 12288), filter_max_tokens = max(Q["filter_max_tokens"], 8192))
MAX_SEQ_LEN, GPU_UTIL, LORA_RANK = Q["max_seq_len"], Q["gpu_util"], Q["lora_rank"]
from recipe_utils import resolve_init_model, sft_repo_name
INIT_MODEL = None if (RUN_SFT and INIT_MODEL == "auto") else resolve_init_model(INIT_MODEL, HF_USER, MODEL_NAME)
if INIT_MODEL and not RUN_SFT:
    from recipe_utils import read_handoff_meta, init_model_problem
    _init_meta = read_handoff_meta(INIT_MODEL, os.environ.get("HF_TOKEN"))
    _init_problem = init_model_problem(_init_meta)
    print("起点检查点的来源记录 (handoff.json):", _init_meta)
    if _init_problem and not ALLOW_UNVERIFIED_INIT:
        raise RuntimeError(f"起点检查点 {INIT_MODEL} 不能确认是正式训练的结果：{_init_problem}。"
                           "先用 notebook 15（MODE='a100'）把 SFT 训完并推送；确实要用这个检查点就设 ALLOW_UNVERIFIED_INIT = True。")
if not RUN_SFT and not INIT_MODEL:
    print("[!] RUN_SFT=False 且没有 INIT_MODEL：会直接在 base 上做 GRPO（base 不会收尾，基本学不动）。先跑 notebook 15。")
if PUSH_FINAL_REPO == "auto":
    PUSH_FINAL_REPO = sft_repo_name(HF_USER, MODEL_NAME).replace("-s1-sft", f"-{EXP_NAME.lower()}-grpo")
from recipe_utils import persist_dir, StageCache, read_progress, write_progress
PERSIST = persist_dir(f"{EXP_NAME}_{MODE}", USE_DRIVE)     # 开头会弹一次 Drive 授权
CACHE = StageCache(PERSIST, enabled = RESUME)
CACHE.check_config(dict(model = MODEL_NAME, init_model = INIT_MODEL, run_sft = RUN_SFT, chat_style = CHAT_STYLE, grpo_data = GRPO_DATA,
                        grpo_filter = GRPO_FILTER, curriculum_rounds = CURRICULUM_ROUNDS, grpo_lr = GRPO_LR,
                        grpo_max_completion = GRPO_MAX_COMPLETION, epsilon_high = EPSILON_HIGH, format_weight = FORMAT_WEIGHT,
                        **{k: Q[k] for k in ("max_seq_len", "lora_rank", "grpo_steps", "grpo_prompts_per_step", "num_gen",
                                             "filter_k", "filter_max_tokens", "val_n", "eval_n", "eval_max_tokens")}))
def ADIR(name):
    return f"{PERSIST}/{name}"
print(EXP_NAME, MODE, "start from:", INIT_MODEL or MODEL_NAME, "-> push:", PUSH_FINAL_REPO, CHAT_STYLE, Q)
print("persist dir:", PERSIST, "| finished stages:", CACHE.done())
# 上一次运行最后在做什么（progress.json）：notebook 的输出经常比运行本身早停，这个文件才是它实际跑到哪一步的记录
print("last progress in this folder:", read_progress(PERSIST))

handoff.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

起点检查点的来源记录 (handoff.json): {'notebook': '15_sft_checkpoint', 'exp': 'S1', 'mode': 'a100', 'variant': 's1', 'model': 'unsloth/Qwen3-4B-Base', 'steps': 3407, 'total_steps': 3407, 'finished': True, 'loss_last': 0.5548}
Mounted at /content/drive
R2 a100 start from: darcyyao/qwen3-4b-s1-sft -> push: darcyyao/qwen3-4b-r2-grpo olmo {'max_seq_len': 12288, 'gpu_util': 0.8, 'lora_rank': 64, 'sft_dolci_n': 20000, 'sft_mcq_n': 2000, 'sft_max_tokens': 6144, 'sft_steps': -1, 'sft_lr': 0.0001, 'sft_batch': 4, 'sft_accum': 4, 'grpo_steps': 120, 'grpo_prompts_per_step': 16, 'num_gen': 8, 'grpo_max_completion': 6144, 'filter_k': 4, 'filter_max_tokens': 3072, 'val_n': 400, 'eval_configs': ['gpqa_diamond', 'gpqa_main'], 'eval_n': None, 'eval_max_tokens': 8192, 'eval_samples': 8, 'shuffle_buffer': 20000}
persist dir: /content/drive/MyDrive/gpqa_ckpt/R2_a100 | finished stages: []


## 1. 加载模型（`fast_inference=True` = 训练和生成共用一个 vLLM 引擎）

In [7]:
import os, torch
if torch.cuda.get_device_capability()[0] < 8:   # 再设一次，防止跳过了前面的格
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
from unsloth import FastLanguageModel
from recipe_utils import (RESPONSE_PARTS, STOP_STRS, TABLE_COLS, QuickEval, setup_chat, chat_prompt,
                          log_eval, print_table, sciq_mcq, sciq_format_rows, load_mcq_rows,
                          drsci_numeric_row, decontam_rows, n_chat_tokens, correctness_score, format_score,
                          soft_overlong_penalty, pass_stats, paired_delta, suggest_completion_cap, lora_delta,
                          config_kwargs, wandb_init, load_guard, restore_lora, end_runtime, guard_fast_generate)

# 这一格可以反复运行：模型已经加载过就直接复用（vLLM 引擎不能在同一个进程里建第二次，以前重跑会报错）。
_KEY = (INIT_MODEL or MODEL_NAME, MAX_SEQ_LEN, LORA_RANK, GPU_UTIL, LOAD_IN_4BIT)
if load_guard(globals(), _KEY):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = INIT_MODEL or MODEL_NAME,
        max_seq_length = MAX_SEQ_LEN,
        load_in_4bit = LOAD_IN_4BIT,
        fast_inference = True,
        max_lora_rank = LORA_RANK,
        gpu_memory_utilization = GPU_UTIL,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = LORA_RANK,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = LORA_RANK * 2,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
    _LOADED_KEY = _KEY
else:
    print("[i] 模型已经加载，直接复用（没有重新加载）。训练格开始时会把 LoRA 权重恢复到起点。")
# 中断过的生成不能污染下一次生成：vLLM 被点停止后，没做完的请求留在引擎里，下一次调用会先把它们做完，
# 并把它们的输出排在自己的输出前面（答案和题目就对不上了）。这个包装在每次调用前清掉残留，多出来的输出丢掉。
guard_fast_generate(model)
# 模板 + 结束符（照 OLMo 3：最后一个 assistant 回合以 eos=<|endoftext|> 结束；generation_config 两个结束符都停）
STOP_IDS = setup_chat(tokenizer, model, style = CHAT_STYLE)
EOS_ID = tokenizer.eos_token_id
_demo = tokenizer.apply_chat_template([{"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello"}], tokenize = False)
print(repr(_demo))
assert _demo.endswith(tokenizer.eos_token), "最后一个回合应以 eos 结束"
print("eos:", tokenizer.eos_token, EOS_ID, "| stop ids:", STOP_IDS, "| pad:", tokenizer.pad_token)

RUN_NAME = f"{EXP_NAME}-{MODEL_NAME.split('/')[-1]}-{MODE}"
wandb, REPORT_TO = wandb_init(WANDB_PROJECT, RUN_NAME, Q | dict(
    exp = EXP_NAME, mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL, base_model = BASE_MODEL, chat_style = CHAT_STYLE,
    run_sft = RUN_SFT, run_grpo = RUN_GRPO, dolci = DOLCI_DATASET, dolci_include = DOLCI_INCLUDE, dolci_exclude = DOLCI_EXCLUDE,
    think_n = THINK_N, extra_sft = EXTRA_SFT, sft_select = SFT_SELECT, sft_pool_factor = SFT_POOL_FACTOR,
    coverage_norm = COVERAGE_NORM, grpo_data = GRPO_DATA, grpo_filter = GRPO_FILTER, curriculum_rounds = CURRICULUM_ROUNDS,
    grpo_lr = GRPO_LR, grpo_max_completion = GRPO_MAX_COMPLETION, epsilon_high = EPSILON_HIGH, format_weight = FORMAT_WEIGHT, overlong_cache = OVERLONG_CACHE,
    gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS), WANDB_ENTITY)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1115 07:42:17.638000 435 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 07:42:17.746000 435 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.13.0+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Qwen3 patching. Transformers: 5.17.0. vLLM: 0.31.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not planning a device map; vLLM places its own weights. Using `sequential`.


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.855.
Unsloth: Patched vLLM's piecewise graph-partition pass so compilation_config=3 (CUDA graphs) works with LoRA.
Unsloth: Not an error, but `level` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_inductor` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `swap_space` is not supported in vLLM. Skipping.
Unsloth: Not an error, but `device` is not supported in vLLM. Skipping.
Unsloth: vLLM loading /root/.cache/huggingface/hub/models--darcyyao--qwen3-4b-s1-sft/snapshots/4bcf7a5cf8edbafcb6ee41afc41edd3300f263ab with actual GPU utilization = 84.34%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.49 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 8192. Num Sequences = 96.
Unsloth: vLLM's KV Cache c

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


... (13 outputs left out)

Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).
'<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\nHello<|endoftext|>'
eos: <|endoftext|> 151643 | stop ids: [151643, 151645, 151644, 70564] | pad: <|vision_pad|>


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: darcy-yi-yao (darcy-yi-yao-apple) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


W&B: https://wandb.ai/darcy-yi-yao-apple/gpqa-hackathon/runs/bxa842el


## 2. GPQA 快速评测 + 诊断（只输出聚合数字；不记录题目或模型输出）

`flex` = 输出中最后一个 `(A)`–`(D)`（排行榜指标），`strict` = 第一个 `The answer is (X)`。
每题采样 `eval_samples` 次，`acc_flex` 是所有采样的平均（avg@k），`acc_flex_se` 是它的标准误（单位：分）。
**比较两个阶段时看配对差值**（第 5 节 `EVAL.compare_all()`：同一批题上的差 ± 标准误；|z| < 2 就是看不出差别）。
`gpqa_main` 包含全部 Diamond 题，所以 Main 还会报 `acc_flex_diamond_part`（其中的 Diamond 题）和 `acc_flex_rest`（其余 250 题）。

**分数丢在哪**：每个结果下面多两行。第一行是 `acc_answered`（只算给出了答案字母的回答，反映推理本身）、`acc_stopped`（自己收尾的回答）、
`acc_truncated`（写满预算被截断的回答）；`acc_flex` 和 `acc_answered` 的差就是"没给出答案"丢掉的分。第二行按学科（Physics / Chemistry / Biology）
列出分数、无答案比例和截断比例。配对比较（第 5 节）也按学科各给一个差值。

**起点模型的分数**（阶段 `0_init`）：`REUSE_INIT_EVAL=True` 时先找 notebook 15 对同一个检查点测过的结果（Drive 上 15 的运行目录，
核对训练步数和最终 loss 都一致才用），找到就不再评测起点，后面的评测也用和它一样的采样次数。找不到就按 `RUN_INIT_EVAL` 评测。

诊断指标：`repeat_rate`（某 8 词片段重复 ≥4 次，打转）、`answer_then_more`（答案之后还写了 ≥30% 的内容）、`glitch_rate`（出现 X1 里的乱码 token `\tNdrFc`）。

In [8]:
import random, json, statistics, time
from collections import Counter
from datasets import load_dataset
from vllm import SamplingParams

EVAL = QuickEval.from_gpqa(tokenizer, Q["eval_configs"], Q["eval_n"], max_tokens = Q["eval_max_tokens"],
                           max_model_len = MAX_SEQ_LEN, samples = Q["eval_samples"], stop_ids = STOP_IDS)
RESULTS = EVAL.results

def quick_eval(tag, lora_request = None, samples = None):
    write_progress(PERSIST, f"gpqa_eval:{tag}")
    r = EVAL.run(tag, lambda texts, kw: model.fast_generate(texts, sampling_params = SamplingParams(**kw),
                                                             lora_request = lora_request), samples, cache = CACHE)
    log_eval(wandb, r, tag)

def diagnose(tag, lora_request = None, k = 3):
    # 看清楚答案后面生成了什么（SciQ 验证集，公开数据；不跳过特殊 token）
    val = load_dataset("allenai/sciq", split = "validation").select(range(k))
    prompts = [chat_prompt(tokenizer, sciq_mcq(ex, 10_000 + i)[0]) for i, ex in enumerate(val)]
    sp = SamplingParams(temperature = 0.0, max_tokens = 512, stop_token_ids = STOP_IDS, stop = STOP_STRS,
                        skip_special_tokens = False)
    for o in model.fast_generate(prompts, sampling_params = sp, lora_request = lora_request):
        c = o.outputs[0]
        print(f"--- [{tag}] finish={c.finish_reason} stop={c.stop_reason!r} tokens={len(c.token_ids)}")
        print(repr(c.text[-300:]))

def stop_check(rows, k = 16):
    # teacher forcing：模型在训练样本答案结尾预测 eos 的概率（应接近 1）；以及输出层几行的范数
    os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    glitch = tokenizer.convert_tokens_to_ids("ĉNdrFc")
    out = {}
    try:
        p_eos, p_im_end, p_glitch = [], [], []
        for r in rows[:k]:
            ids = tokenizer(tokenizer.apply_chat_template(r["messages"], tokenize = False), return_tensors = "pt",
                            add_special_tokens = False).input_ids[:, -2048:].to(model.device)
            if ids[0, -1].item() != EOS_ID:
                continue
            with torch.no_grad():
                probs = torch.softmax(model(input_ids = ids[:, :-1]).logits[0, -1].float(), -1)
            p_eos.append(probs[EOS_ID].item()); p_im_end.append(probs[im_end].item()); p_glitch.append(probs[glitch].item())
        out = dict(n = len(p_eos), p_eos_mean = round(statistics.fmean(p_eos), 4) if p_eos else None,
                   p_eos_median = round(statistics.median(p_eos), 4) if p_eos else None,
                   p_im_end_mean = round(statistics.fmean(p_im_end), 4) if p_im_end else None,
                   p_glitch_mean = round(statistics.fmean(p_glitch), 4) if p_glitch else None)
        W = model.get_output_embeddings().weight
        norms = W.float().norm(dim = -1)
        out.update(norm_eos = round(norms[EOS_ID].item(), 3), norm_im_end = round(norms[im_end].item(), 3),
                   norm_glitch = round(norms[glitch].item(), 3), norm_mean = round(norms.mean().item(), 3))
    except Exception as e:
        print("stop check skipped:", type(e).__name__, e)
    finally:
        os.environ.pop("UNSLOTH_RETURN_LOGITS", None)
    print("stop check:", out)
    return out

# 起点模型的 GPQA 分数（阶段 "0_init"）：先找 notebook 15 测过的结果（REUSE_INIT_EVAL），没有就按 RUN_INIT_EVAL 决定现在测还是训完再测。
# 逻辑在 scripts/grpo_run.py 的 init_eval 里。
from grpo_run import init_eval
INIT_EVAL = init_eval(globals())
print("starting eval:", INIT_EVAL)

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

gpqa_diamond.csv:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/198 [00:00<?, ? examples/s]

gpqa_main.csv:   0%|          | 0.00/3.21M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/448 [00:00<?, ? examples/s]

(MainProcess pid=435) INFO 11-15 07:46:11 [scheduler.py:2691] setting pause state to UNPAUSED


Rendering prompts:   0%|          | 0/198 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1584 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(MainProcess pid=435) WARNING 11-15 07:46:13 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts:  12%|█▏        | 184/1584 [01:19<07:40,  3.04it/s, est. speed input: 543.49 toks/s, output: 1733.88 toks/s]

KeyboardInterrupt: 

## 3. SFT（chat 模板转成 `text` + `train_on_responses_only`）

1. **Dolci-Instruct** 按来源 / domain 过滤到科学、数学（不要逻辑题），长度 ≤ `sft_max_tokens`。
2. 可选 **Dolci-Think** 思维链（`THINK_N`）和 **额外数据源**（`EXTRA_SFT`）。
3. **覆盖度选择**（`SFT_SELECT="coverage"`）：先抽 `SFT_POOL_FACTOR` 倍候选，再按 Dr.SCI 的 EESFT 贪心挑出最能扩大 4-gram 覆盖的样本。
4. **选择题格式样本**：SciQ 训练集套 GPQA 模板，答案 = `support` 段落 + `The answer is (X).`。

全部过一遍 GPQA 13-gram 去污染。

In [9]:
os.makedirs("data", exist_ok = True)
N_DOLCI, SFT_MAX, SHUF = Q["sft_dolci_n"], Q["sft_max_tokens"], Q["shuffle_buffer"]
POOL = SFT_POOL_FACTOR if SFT_SELECT == "coverage" else 1
inc = " ".join(DOLCI_INCLUDE); exc = " ".join(DOLCI_EXCLUDE)
inc_arg = f"--include {inc}" if inc else ""
N_POOL = N_DOLCI * POOL
if RUN_SFT:
    !python scripts/prepare_dolci_sft.py --dataset {DOLCI_DATASET} --out data/dolci.jsonl --n {N_POOL} --tokenizer {MODEL_NAME} --max-tokens {SFT_MAX} --shuffle-buffer {SHUF} {inc_arg} --exclude {exc}
    if THINK_N > 0:
        t_inc = " ".join(THINK_INCLUDE); N_THINK_POOL = THINK_N * POOL
        !python scripts/prepare_dolci_sft.py --dataset {THINK_DATASET} --out data/think.jsonl --n {N_THINK_POOL} --tokenizer {MODEL_NAME} --max-tokens {SFT_MAX} --shuffle-buffer {SHUF} --include {t_inc} --exclude {exc}

In [10]:
DATA = {}
if RUN_SFT:
    from recipe_utils import prepare_sft_sources
    from select_coverage import select as coverage_select, hashed_coverage

    def _load(path):
        return [json.loads(l) for l in open(path)] if os.path.exists(path) else []

    pool = [{"messages": r["prompt"] + r["completion"], "source": r["source"], "n_tokens": r["n_tokens"], "part": "dolci"}
            for r in _load("data/dolci.jsonl")]
    pool += [{"messages": r["prompt"] + r["completion"], "source": r["source"], "n_tokens": r["n_tokens"], "part": "think"}
             for r in (_load("data/think.jsonl") if THINK_N > 0 else [])]
    if EXTRA_SFT:
        pool += [dict(r, part = "extra") for r in prepare_sft_sources(EXTRA_SFT, MODEL_NAME, SFT_MAX, scale = POOL, shuffle_buffer = SHUF)]
    pool, dropped = decontam_rows(pool, lambda r: {"messages": r["messages"]}, "sft_pool")
    target = N_DOLCI + (THINK_N if THINK_N > 0 else 0) + sum(s["n"] for s in EXTRA_SFT)
    texts = ["\n".join(m["content"] for m in r["messages"]) for r in pool]
    if SFT_SELECT == "coverage" and len(pool) > target:
        t0 = time.time()
        idx = coverage_select(texts, target, norm = COVERAGE_NORM)
        rand = random.Random(0).sample(range(len(pool)), min(target, len(pool)))
        cov = dict(selected_4grams = hashed_coverage([texts[i] for i in idx]),
                   random_4grams = hashed_coverage([texts[i] for i in rand]), minutes = round((time.time() - t0) / 60, 1))
        print("coverage selection:", cov)
    else:
        idx, cov = list(range(min(target, len(pool)))), {}
    chosen = [pool[i] for i in idx]
    sft_mcq = sciq_format_rows(Q["sft_mcq_n"])
    sft_rows = [{"messages": r["messages"]} for r in chosen] + sft_mcq
    random.Random(0).shuffle(sft_rows)
    DATA = dict(pool = len(pool), selected = len(chosen), sciq_format = len(sft_mcq), dropped_decontam = dropped,
                select = SFT_SELECT, coverage = cov, tokens = sum(r["n_tokens"] or 0 for r in chosen),
                by_part = Counter(r["part"] for r in chosen).most_common(),
                sources = Counter(r["source"] for r in chosen).most_common(10))
    wandb.config.update({"data": DATA}, allow_val_change = True)
    print(DATA)

In [11]:
SFT, STOP = {}, {}
if RUN_SFT:
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from unsloth.chat_templates import train_on_responses_only

    # 文本以 eos 结束（模板最后一个回合写的就是 eos），TRL 不会再额外追加
    sft_ds = Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize = False)} for r in sft_rows])
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = sft_ds,
        args = SFTConfig(
            dataset_text_field = "text",
            max_length = MAX_SEQ_LEN,
            per_device_train_batch_size = Q["sft_batch"],
            gradient_accumulation_steps = Q["sft_accum"],
            warmup_steps = 10,
            max_steps = Q["sft_steps"],
            num_train_epochs = 1,
            learning_rate = Q["sft_lr"],
            logging_steps = 5,
            optim = "adamw_8bit",
            weight_decay = 0.001,
            lr_scheduler_type = "linear",
            seed = 3407,
            report_to = REPORT_TO,
            run_name = RUN_NAME + "-sft",
            output_dir = "outputs/sft",
            save_strategy = "no",
        ),
    )
    instr_part, resp_part = RESPONSE_PARTS[CHAT_STYLE]
    trainer = train_on_responses_only(trainer, instruction_part = instr_part, response_part = resp_part)
    # 诊断：训练标签的最后一个 token 应该是 eos（模型要学会写它）
    try:
        _labels = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
        print("label tail:", repr(tokenizer.decode(_labels[-12:])), "| ends with eos:", _labels[-1] == EOS_ID)
    except Exception as e:
        print("label check skipped:", e)

    print("fresh LoRA:", restore_lora(model))     # 重跑这一格时从头开始
    sft_stats = trainer.train()
    sft_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    SFT = dict(steps = trainer.state.global_step, minutes = round(sft_stats.metrics["train_runtime"] / 60, 1),
               loss_first = sft_losses[0] if sft_losses else None, loss_last = sft_losses[-1] if sft_losses else None)
    print(SFT)
    STOP = stop_check(sft_rows)
    model.save_lora(ADIR("sft_lora"))
    quick_eval("1_sft", model.load_lora(ADIR("sft_lora")))
    diagnose("1_sft", model.load_lora(ADIR("sft_lora")))
    if PUSH_SFT_REPO:
        try:
            model.push_to_hub_merged(PUSH_SFT_REPO, tokenizer, save_method = "merged_16bit",
                                     private = True, token = os.environ.get("HF_TOKEN"))
        except TypeError:   # 旧版 Unsloth 没有 private 参数：先在 HF 网页上建好私有仓库再推
            model.push_to_hub_merged(PUSH_SFT_REPO, tokenizer, save_method = "merged_16bit",
                                     token = os.environ.get("HF_TOKEN"))
        print("SFT pushed to", PUSH_SFT_REPO, "→ notebook 12 的 INIT_MODEL 填它")
    del sft_ds, trainer
    torch.cuda.empty_cache(); import gc; gc.collect()

## 4. GRPO（`vllm_sampling_params` + 多个奖励函数）

**代码在 `scripts/grpo_run.py`**（题目池、留出验证集、预筛、奖励函数、分段训练、检查点和续训、阶段验收），下面两格只有开关和两次调用。
第 3 格每次启动都会 `git pull`，所以就算这个 notebook 是很早以前打开的副本，跑的也是仓库里最新的逻辑；从旧副本"保存到 GitHub"也改不掉它。

**题库**（`GRPO_DATA`）：
- `science`（默认）：50% **MegaScience 留出的 4 选 1 科学题**（NVIDIA OpenScience 的研究生水平选择题，经 MegaScience 的 LLM 去污染和难度筛选；
  和 GPQA 同一种题面；notebook 15 的 SFT 只用了另外 55% 的题，这里的题没见过解答）+ 20% MMLU-Pro STEM + 30% Dr.SCI 数值题。
- `mix`（X1b 用的）：一半 **MMLU-Pro STEM**（物理/化学/生物/工程/数学，抽成 4 选 1，套 GPQA 模板，答案看最后一个 `(A)`–`(D)`），
  一半 **Dr.SCI 数值题**（[MiniByte-666/Dr.SCI](https://huggingface.co/datasets/MiniByte-666/Dr.SCI) 的可验证题里参考答案是数字的；用它自带的 `\boxed{}` 提示，相对误差 ≤1% 算对；没有 25% 的猜中率）。
- 课程（`CURRICULUM_ROUNDS=2`）：第一段 Dr.SCI 用 Qwen3-32B 通过率 0.5–0.875 的题，第二段用 0.25–0.375 的题；每段开始前用**当前模型**每题采样 4 次，只留答对 1–3 次的（`GRPO_FILTER`）。

**奖励**（三个函数相加；每条回答的总分见下表）：
- `correctness_reward`：答对 +2，答错 0，找不到答案 −1
- `format_reward`：答案以规定形式恰好出现一次 +0.5（选择题 `The answer is (X)`，数值题 `\boxed{}`）
- `overlong_reward`：DAPO 软超长惩罚——长度进入上限前的最后一段后线性扣到 −1，被截断 −1

| 回答的结果 | 总分 |
|---|---|
| 正常收尾且答对 | +2 到 +2.5（贴近上限时最低 +1） |
| 正常收尾但答错 | 0 到 +0.5 |
| 正常收尾但没给出答案 | −1 |
| 写满上限（被截断），不管内容 | −2 |

**奖励设计的三个结论**（依据：TMax、OpenThoughts-Agent、Kimi K3、DAPO、DeepSeek-V3.2、ToolRL）：
1. **不加 KL（`GRPO_BETA = 0`）**。TMax（β = 0）、OpenThoughts（无 KL）、DAPO（去掉 KL）都不用，DeepSeek-V3.2 也写明数学类推理用弱 KL 或不用更好。
   这些工作靠别的手段限制每步更新：裁剪（`EPSILON_HIGH`、`MAX_GRAD_NORM`）、更大的组、屏蔽偏离过大的 token。R1 的崩溃是奖励在给"不收尾"发正分，KL 只会让它崩得慢一点。
2. **不按长度扣分，只设预算**。ToolRL 的消融里，给长度加奖励在小模型上明显降分；Kimi K3 的做法是按冷启动模型自己的长度定预算，超出预算的回答直接记最低分。
   这里的预算就是回答上限（模型自己收尾长度 p99 的 2 倍，只降不升）：预算内长短不影响得分，贴近上限的最后一段线性扣分（DAPO），写满上限记最低分。
3. **截断的回答不进损失，只进基线**（`MASK_TRUNCATED = True`，DAPO 的 overlong filtering）。截断的回答每条有几千个 token，按 token 平均的损失会被它们主导，
   而且会连带惩罚前面本来合理的推理。它们的 −2 仍然计入组内平均，所以同组里正常收尾的回答得到正的优势，"收尾"是通过这条路径被强化的。
   代价：如果一步里所有回答都被截断，这一步没有梯度；这种情况由下面的提前停止兜住。

**优势的归一化（`SCALE_REWARDS = "batch"`）**：TRL 默认按每道题的组内标准差缩放，这样一组里只有格式分或超长扣分有差别时，这点差别会被放大到和"对 / 错"一样大。
改成按整步所有回答的标准差缩放后，各项保持上表的相对大小，正确分占主导（ToolRL：正确分的权重要明显大于格式分；TMax 只做中心化）。R1 用的是 `"group"`，想对照可以改回去。

**怎么判断 GRPO 有没有用**（X1b 的教训：别只看 GPQA 上一次采样的分数）：
1. **留出验证集** `rl_val`：和训练题同分布、但不参与训练的 `val_n` 道题，每题采样 4 次（温度 1.0，和 rollouts 一样）。训练前、每段之后各测一次，
   并给出配对差值 ± 标准误。**它不涨 = GRPO 没学到能泛化的东西**（步数 / 学习率 / 题量的问题）；它涨了但 GPQA 不涨 = 迁移的问题（题库和 GPQA 差太远）。
2. `adapter_change`：这一段训练让 LoRA 权重相对上一段变了多少（相对范数）。X1b 的学习率 5e-6 × 100 步，每个参数最多移动 5e-4。
3. W&B 上的 `rewards/correctness_reward/mean`：每步是不同的题，单步噪声很大，要看 20 步以上的平均。

**每步的规模**：`grpo_prompts_per_step` 道题 × `num_gen` 个回答。X1b 是 4 × 8（梯度噪声大，100 步只见 400 道题）；现在 a100 是 16 × 8，120 步见 1920 道题。
生成很便宜（vLLM 并行），每步的固定开销（合并 / 还原权重、暂停引擎）占大头，所以加大每步题数比加步数划算。

**稳定性保护（R1 第 2 段把模型训到不再收尾之后加的）**：
- 写满上限的回答不给正确分和格式分（之前"截断但出现过正确字母"比"收尾但答错"得分高），也不参与梯度。
- 截断比例连续 3 步 ≥ 0.15 就提前结束这一段；一段训完后在留出集上检查，正常收尾率 < 90% 或通过率显著下降就作废，退回上一段。
  两个阈值都以起点模型为准：起点自己的收尾率不到 95% 时，验收线是"起点 − 5 个点"，提前结束的线是"起点的不收尾比例 + 0.10"。
- 回答上限只降不升；第 2 段起学习率减半，每段都预热。
- 每 5 步把 adapter 存到 Drive（`grpo_ckpt_s<段>/`，保留最近 3 个），断线后 Run all 从最近的检查点接着训这一段；最新的读不出来就用前一个。
- 训练被手动中断或报错（显存不足、CUDA 错误）时，先把内存里的 adapter 存成一个检查点再停。Colab 断线或计算单元用完时 runtime 直接消失，这一步来不及执行，靠的是上面的定期存盘。
- 不合格的阶段，它的检查点也改名留档（`grpo_ckpt_s<段>_rejected_<时间>`），可以手动从崩溃之前的某一步接着训。
- 最终评测和推送只用最后一个合格阶段的 adapter。


In [ ]:
# 第 4 节的逻辑都在 scripts/grpo_run.py 里：题目池、留出验证集、预筛、奖励函数、每步诊断、分段训练、检查点和续训、阶段验收。
# 第 3 格每次都会 git pull，所以即使这个 notebook 是以前打开的旧副本，跑的也是最新的逻辑；这里和下一格只放开关。
from grpo_run import GrpoRun
GRUN = GrpoRun(globals()) if RUN_GRPO else None
if GRUN:
    GRUN.prepare()                                  # 题目池 + 留出验证集（约 3 分钟：第一次要下载 MegaScience）
RL_VAL = GRUN.rl_val_results if GRUN else {}        # 留出验证集的结果：before_grpo / after_s0 / after_s1


In [ ]:
GRPO = {}
GRPO_FINAL = None      # 最后一个被接受的阶段的 adapter 目录名；第 6 节合并 / 推送的就是它
# ---- 稳定性和存盘设置（改这里不会让已完成阶段的缓存失效）----
CKPT_EVERY     = 5     # 每 N 步把 adapter 存到 Drive；断线后 Run all 从最近的检查点接着训这一段（最多丢 N 步）
CKPT_KEEP      = 3     # 每次启动保留最近几个检查点（a100 的 adapter 每个几百 MB）
CLIP_PATIENCE  = 3
LATER_STAGE_LR = 0.5   # 第 2 段起学习率乘这个系数（R1 的第 2 段用满学习率、没有预热，6 步后开始发散）
GRPO_BETA      = 0.0   # KL 惩罚系数（相对起点模型）。0 = 不加：TMax / OpenThoughts / DAPO 都是 0，DeepSeek-V3.2 对推理题也用弱 KL 或不用。
                       # 只有 diag/ 曲线显示奖励修好后仍然漂移时再试 0.01–0.04（每步多一次前向，会变慢）
SCALE_REWARDS  = "batch"   # 优势按整步所有回答的标准差缩放，格式分 / 超长扣分保持设计的相对大小；"group" = TRL 默认（R1 用的），按每道题的组缩放
MASK_TRUNCATED = True  # 被截断的回答不进损失（DAPO overlong filtering），它们的 −2 只压低组内基线；False = 直接对它们做负梯度（R1 的设置）
MAX_GRAD_NORM  = 1.0   # 梯度裁剪；训练日志里 grad_norm 经常顶到这个值就调小（例如 0.3）
MIN_STOP_RATE  = 90.0  # 一段训完后，留出集上正常收尾的比例低于它：这一段作废，退回上一段的 adapter。起点模型自己不到 95% 时，改用"起点 − 5 个点"
CLIP_STOP      = 0.15  # 截断比例（completions/clipped_ratio）连续 CLIP_PATIENCE 步 >= 这个值：提前结束这一段。起点模型不收尾的比例高时，改用"起点 + 0.10"
if RUN_GRPO:
    # 开关在用到的时候才读：改了上面的值重跑这一格就生效。每次运行都从起点开始；已经完成并合格的阶段直接读回，
    # 训到一半的阶段从 Drive 上最新的检查点接着训。训练被中断（手动停止 / 显存不足 / CUDA 报错）时先把内存里的 adapter 存盘再停。
    GRPO = GRUN.train()
    GRPO_FINAL = GRUN.final_adapter


## 5. 汇总（W&B 上也有同样的表；最后打印的 JSON 可以贴回给 Claude）

In [ ]:
rows = print_table(RESULTS)
print("\n阶段之间的配对比较（同一批题，后一阶段 − 前一阶段，单位：分）：")
COMPARE = EVAL.compare_all()
if RL_VAL:
    print("\n留出验证集通过率（GRPO 学到东西没有）：", {k: (v.get("pass_rate"), v.get("pass_by_type")) for k, v in RL_VAL.items()})
print("SFT:", SFT); print("STOP:", STOP); print("GRPO:", GRPO)
REPORT = dict(notebook = "11_sft_grpo_unsloth", exp = EXP_NAME, mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL,
              chat_style = CHAT_STYLE, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, q = Q, dolci = DOLCI_DATASET,
              dolci_include = DOLCI_INCLUDE, think_n = THINK_N, extra_sft = EXTRA_SFT, sft_select = SFT_SELECT,
              grpo_data = GRPO_DATA, grpo_filter = GRPO_FILTER, curriculum_rounds = CURRICULUM_ROUNDS, grpo_lr = GRPO_LR,
              epsilon_high = EPSILON_HIGH, data = DATA, sft = SFT, stop_check = STOP, grpo = GRPO, quick_eval = RESULTS,
              compare = COMPARE, versions = VERSIONS, wandb = wandb.run.url if REPORT_TO == "wandb" else None)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open(f"outputs/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
if REPORT_TO == "wandb":
    wandb.log({"gpqa_quick_eval": wandb.Table(columns = TABLE_COLS, data = rows)})
    art = wandb.Artifact(f"report-{RUN_NAME}", type = "report")
    art.add_file(f"outputs/report_{EXP_NAME}.json")
    wandb.log_artifact(art)
    wandb.finish()
json.dump(REPORT, open(f"{PERSIST}/report_{EXP_NAME}.json", "w"), indent = 1, default = str)   # 断开之后也能在 Drive 里找到
write_progress(PERSIST, "report_written")
print("=" * 30, "把下面整段 JSON 贴回给 Claude（也存在 Drive：", f"{PERSIST}/report_{EXP_NAME}.json", "）", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False, default = str))

## 6. 合并保存 → 推到 Hub → 用 `main` 分支的 `01` 做正式评测

默认合并最后一个阶段并推到 HF 私有仓库（`PUSH_FINAL_REPO="auto"`）；推送失败就存到持久目录。第 5 节的配对比较如果显示最后一段没有提升，
把 `FINAL_ADAPTER` 设成 `"grpo_lora_s0"` 再跑这一格；如果 GRPO 整体没有提升，直接用起点（S1 / DPO 模型）去做正式评测，不用再推。

`merged_16bit` 把 LoRA 合并进权重（排行榜要求）；保存的 tokenizer 带着本 notebook 的模板和 eos=`<|endoftext|>`，`generation_config` 里两个结束符都在，锁定评测会正确停止。
推到 Hub 后**换一个新的 runtime** 打开 [`01_eval_gpqa.ipynb`](https://colab.research.google.com/github/yy726/GPQA-eval/blob/main/notebooks/01_eval_gpqa.ipynb)（生成预算 8192）。A100 上训练的模型提交时在 `notes` 里注明 A100（不参与排名）。

In [ ]:
if FINAL_ADAPTER:
    # 改为合并较早的阶段（例如第 5 节的配对比较显示 GRPO 没有提升时）
    print("merging adapter:", FINAL_ADAPTER, restore_lora(model, ADIR(FINAL_ADAPTER)))
_final_name = FINAL_ADAPTER or globals().get("GRPO_FINAL")
_final_key = f"pushed_{_final_name or 'last'}"          # 按 adapter 记录：换了最终 adapter 会重新推送
if RUN_GRPO and not _final_name:
    print("[!] 没有合格的 GRPO adapter（所有阶段都被判为不合格）：不合并、不推送。Hub 上已有的模型不会被覆盖。")
elif CACHE.has(_final_key):
    print("[i] 已经推送 / 保存过：", CACHE.load(_final_key))
else:
    _where = PUSH_FINAL_REPO
    try:
        if not PUSH_FINAL_REPO:
            raise ValueError("PUSH_FINAL_REPO is None")
        try:
            model.push_to_hub_merged(PUSH_FINAL_REPO, tokenizer, save_method = "merged_16bit", private = True,
                                     token = os.environ.get("HF_TOKEN"))
        except TypeError:   # 旧版 Unsloth 没有 private 参数
            model.push_to_hub_merged(PUSH_FINAL_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
        print(f"pushed (private): https://huggingface.co/{PUSH_FINAL_REPO}（正式评测前改成公开）")
    except Exception as e:
        _where = ADIR("merged_16bit")
        print("[!] 没有推到 HF：", type(e).__name__, str(e)[:200], "→ 合并模型存到", _where)
        model.save_pretrained_merged(_where, tokenizer, save_method = "merged_16bit")
    CACHE.save(_final_key, {"model": _where})

## 7. 断开 runtime

adapter、合并模型、报告都已经在 Google Drive / HF 上。`AUTO_DISCONNECT=True` 时这一格在 60 秒后释放 runtime，不再消耗 CU——
所以可以 **Run all 之后离开**：中途断线就重新打开、再 Run all，已经完成的评测和 GRPO 阶段会直接跳过。

In [ ]:
end_runtime(AUTO_DISCONNECT)